In [1]:
# Neural Identifier Training with Particle Filters - Lorenz Attractor
# Neural Identifier Training with Particle Filters - Lorenz System (Chaotic)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1) True nonlinear system (Lorenz Attractor)
# ============================================================
def plant_dynamics(x_state, u=0):
    """
    Continuous dynamics for Lorenz System.
    x_state = [x, y, z]
    
    Equations:
    dx/dt = sigma * (y - x)
    dy/dt = x * (rho - z) - y
    dz/dt = x * y - beta * z
    
    Standard Chaotic Params: sigma=10, rho=28, beta=8/3
    """
    sigma = 10.0
    rho = 28.0
    beta = 8.0 / 3.0
    
    x, y, z = x_state
    
    dx_dt = sigma * (y - x)
    dy_dt = x * (rho - z) - y
    dz_dt = x * y - beta * z
    
    return np.array([dx_dt, dy_dt, dz_dt])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-2):
    """
    One Euler step of the discrete plant with process noise.
    For Lorenz, RK4 is usually better, but for small dt Euler works for demo.
    """
    # Simple Euler integration
    x_dot = plant_dynamics(x_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure (Updated for 3 Inputs) - CONFIGURABLE
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -100, 100) # Clip to avoid overflow
    return 1.0 / (1.0 + np.exp(-beta * z))

# ============================================================
# RHONN EQUATION CONFIGURATIONS
# ============================================================
# Define different RHONN basis function configurations
# Each configuration returns a feature vector z

def rhonn_config_1(x_est, scale=0.1):
    """
    Configuration 1: Second-order sigmoid interactions + quadratic terms
    Good for capturing cross-product dynamics (x*y, x*z, etc.)
    """
    s1 = sigmoidal(x_est[0] * scale)
    s2 = sigmoidal(x_est[1] * scale)
    s3 = sigmoidal(x_est[2] * scale)
    
    return np.array([
        s1*s2, s1*s3, s2*s3,          # Second order interactions
        s1**2, s2**2, s3**2,          # Quadratic self-terms
    ])

def rhonn_config_2(x_est, scale=0.1):
    """
    Configuration 2: First-order sigmoids + linear terms + bias
    Simpler, more interpretable structure
    """
    s1 = sigmoidal(x_est[0] * scale)
    s2 = sigmoidal(x_est[1] * scale)
    s3 = sigmoidal(x_est[2] * scale)
    
    return np.array([
        s1, s2, s3,                    # First order sigmoids
        x_est[0], x_est[1], x_est[2],  # Linear terms
        1.0                            # Bias
    ])

def rhonn_config_3(x_est, scale=0.1):
    """
    Configuration 3: Full combination - all orders
    Most expressive but requires more parameters
    """
    s1 = sigmoidal(x_est[0] * scale)
    s2 = sigmoidal(x_est[1] * scale)
    s3 = sigmoidal(x_est[2] * scale)
    
    return np.array([
        s1, s2, s3,                    # First order
        s1*s2, s1*s3, s2*s3,          # Second order interactions
        s1**2, s2**2, s3**2,          # Quadratic self-terms
        x_est[0], x_est[1], x_est[2], # Linear terms
        1.0                            # Bias
    ])

def rhonn_config_4(x_est, scale=0.1):
    """
    Configuration 4: Compact - interactions + cubics
    Focus on key nonlinear terms for Lorenz
    """
    s1 = sigmoidal(x_est[0] * scale)
    s2 = sigmoidal(x_est[1] * scale)
    s3 = sigmoidal(x_est[2] * scale)
    
    return np.array([
        s1*s2, s1*s3, s2*s3,          # Second order interactions
        s2**3                          # Cubic term for extra expressiveness
    ])

def rhonn_config_custom(x_est, scale=0.1):
    """
    Configuration CUSTOM: Define your own basis functions here
    Example template - modify as needed
    """
    s1 = sigmoidal(x_est[0] * scale)
    s2 = sigmoidal(x_est[1] * scale)
    s3 = sigmoidal(x_est[2] * scale)
    
    # MODIFY THIS ARRAY WITH YOUR DESIRED BASIS FUNCTIONS
    return np.array([
        s1*s2,           # x-y interaction
        s1*s3,           # x-z interaction
        s2*s3,           # y-z interaction
        s1**2,           # x squared term
        s2**2,           # y squared term
        s3**2,           # z squared term
        # Add more terms as needed, e.g.:
        # x_est[0],      # Linear x
        # x_est[1],      # Linear y
        # x_est[2],      # Linear z
        # s1*s2*s3,      # Third order interaction
        # 1.0            # Bias term
    ])

# Dictionary of available configurations
RHONN_CONFIGS = {
    1: {'func': rhonn_config_1, 'name': 'Interactions + Quadratic', 'n_features': 6},
    2: {'func': rhonn_config_2, 'name': 'Sigmoids + Linear + Bias', 'n_features': 7},
    3: {'func': rhonn_config_3, 'name': 'Full Combination', 'n_features': 13},
    4: {'func': rhonn_config_4, 'name': 'Compact Interactions + Cubic', 'n_features': 4},
    'custom': {'func': rhonn_config_custom, 'name': 'Custom Configuration', 'n_features': 6},
}

# Global variable to store selected configuration
SELECTED_RHONN_CONFIG = None

def set_rhonn_config(config_id=1):
    """
    Set the active RHONN configuration
    
    Parameters:
    -----------
    config_id : int or str
        Configuration ID (1, 2, 3, 4, or 'custom')
    """
    global SELECTED_RHONN_CONFIG
    if config_id not in RHONN_CONFIGS:
        raise ValueError(f"Invalid config_id: {config_id}. Choose from {list(RHONN_CONFIGS.keys())}")
    
    SELECTED_RHONN_CONFIG = config_id
    config = RHONN_CONFIGS[config_id]
    print(f"\n{'='*60}")
    print(f"🔧 RHONN Configuration Set:")
    print(f"   ID: {config_id}")
    print(f"   Name: {config['name']}")
    print(f"   Number of features: {config['n_features']}")
    print(f"{'='*60}\n")
    
    return config['n_features']

def construct_z_vector(x_est, scale=0.1):
    """
    Construct feature vector using the selected RHONN configuration
    
    Parameters:
    -----------
    x_est : array
        State estimate [x, y, z]
    scale : float
        Scaling factor for sigmoid inputs
        
    Returns:
    --------
    array : Feature vector
    """
    if SELECTED_RHONN_CONFIG is None:
        raise ValueError("RHONN configuration not set! Call set_rhonn_config() first.")
    
    config_func = RHONN_CONFIGS[SELECTED_RHONN_CONFIG]['func']
    return config_func(x_est, scale)

def print_rhonn_configs():
    """Print all available RHONN configurations"""
    print("\n" + "="*70)
    print("📋 AVAILABLE RHONN CONFIGURATIONS:")
    print("="*70)
    for config_id, config in RHONN_CONFIGS.items():
        print(f"\nConfig {config_id}: {config['name']}")
        print(f"   Features: {config['n_features']}")
        # Show sample output
        sample_state = np.array([1.0, 2.0, 3.0])
        sample_z = config['func'](sample_state, scale=0.1)
        print(f"   Sample z (for x=[1,2,3]): {sample_z[:3]}..." if len(sample_z) > 3 else f"   Sample z (for x=[1,2,3]): {sample_z}")
    print("="*70 + "\n")

def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron.
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-1, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        self.weights, self.P, self.Q, self.R = [], [], [], []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.05
            self.weights.append(w_i)
            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        # Series-Parallel: Use measured true state (chi_k) to build feature vector
        x_state_for_z = np.copy(chi_k) 

        z_i = construct_z_vector(x_state_for_z)          
        H_i = z_i.reshape(-1, 1)                          

        for i in range(self.num_neurons):
            P_pred = self.P[i] + self.Q[i] + np.eye(self.num_weights_per_neuron) * 1e-8
            
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10: M_i = 1e-10

            x_hat_pred_i = self.weights[i] @ z_i
            e_i = chi_kp1[i] - x_hat_pred_i
            e_i = np.clip(e_i, -50.0, 50.0) # Larger clip for Lorenz

            K_i = (P_pred @ H_i).flatten() / M_i

            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.01))
            self.weights[i] += adaptive_eta * K_i * e_i

            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # PSD enforcement
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            if np.min(np.linalg.eigvals(self.P[i])) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-5

# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle Filter trainer for RHONN weights.
    
    Supports different Q_std and R_std values per neuron/state.
    
    Parameters:
    -----------
    num_neurons : int
        Number of neurons (states to estimate)
    num_weights_per_neuron : int
        Number of weights per neuron
    n_particles : int
        Number of particles in the filter
    initial_weights : list of arrays, optional
        Initial weight values for each neuron
    Q_std : float or array-like
        Process noise standard deviation. Can be:
        - Single float: same Q_std for all neurons
        - Array of length num_neurons: different Q_std per neuron
    R_std : float or array-like
        Measurement noise standard deviation. Can be:
        - Single float: same R_std for all neurons
        - Array of length num_neurons: different R_std per neuron
    ess_threshold : float, optional
        Effective Sample Size threshold for resampling
    """
    
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=500,
                 initial_weights=None, Q_std=0.5, R_std=0.5, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Convert Q_std to per-neuron array
        if np.isscalar(Q_std):
            self.Q_std = np.ones(num_neurons) * Q_std
        else:
            self.Q_std = np.array(Q_std)
            if len(self.Q_std) != num_neurons:
                raise ValueError(f"Q_std debe tener {num_neurons} elementos, pero tiene {len(self.Q_std)}")
            
        # Convert R_std to variance per-neuron array
        if np.isscalar(R_std):
            self.R_var = np.ones(num_neurons) * (R_std**2)
            self.R_std = np.ones(num_neurons) * R_std
        else:
            R_std_array = np.array(R_std)
            if len(R_std_array) != num_neurons:
                raise ValueError(f"R_std debe tener {num_neurons} elementos, pero tiene {len(R_std_array)}")
            self.R_std = R_std_array
            self.R_var = R_std_array**2
            
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        # Initialize particle sets and weights for each neuron
        self.particles = []
        self.weights_pf = []

        # Initialize particle sets and weights for each neuron (FULLY VECTORIZED)
        if initial_weights is not None:
            base_weights = np.array([np.copy(initial_weights[i]) if i < len(initial_weights) 
                                    else np.random.randn(num_weights_per_neuron) * 0.05 
                                    for i in range(num_neurons)])
            
            # Shape: (num_neurons, n_particles, num_weights_per_neuron)
            self.particles = (base_weights[:, np.newaxis, :] + 
                            np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.05)
        else:
            self.particles = np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.05

        # Shape: (num_neurons, n_particles)
        self.weights_pf = np.ones((num_neurons, n_particles)) / n_particles

    def _ess(self, w):
        """
        Calculate Effective Sample Size.
        
        Parameters:
        -----------
        w : array
            Particle weights
            
        Returns:
        --------
        float : Effective sample size
        """
        sum_w = np.sum(w)
        if sum_w == 0:
            return 0
        w_norm = w / sum_w
        return 1.0 / np.sum(w_norm**2)

    def _resample_systematic(self, neuron_index):
        """
        Systematic resampling algorithm.
        
        Parameters:
        -----------
        neuron_index : int
            Index of neuron to resample
        """
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]
        
        # Normalize weights
        w = w / np.sum(w)
        N = len(w)
        
        # Systematic resampling
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)
        indexes = np.zeros(N, dtype=int)
        
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j] and j < N - 1:
                j += 1
            indexes[i] = j
            i += 1
        
        # Resample particles and reset weights
        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        Update particle filter with new measurement.
        
        Parameters:
        -----------
        chi_kp1 : array
            Measurement at time k+1 (target output for training)
        chi_k : array
            State at time k (used for feature construction)
        x_hat_previous : array
            Previous estimate (not used in series-parallel mode)
        """
        # Series-Parallel: Use measured true state to build feature vector
        x_state_for_z = np.copy(chi_k)
        z = construct_z_vector(x_state_for_z)

        # Update each neuron independently
        for i in range(self.num_neurons):
            # ===== PREDICT STEP =====
            # Add process noise to particles (weight diffusion)
            # Each neuron uses its own Q_std
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

            # ===== UPDATE STEP =====
            # Compute predictions for all particles
            w_mat = self.particles[i]  # (n_particles, num_weights)
            x_pred_particles = w_mat @ z  # (n_particles,)
            
            # Innovation (prediction error) for each particle
            innov = chi_kp1[i] - x_pred_particles
            
            # Compute likelihood using neuron-specific R_var
            # log-likelihood for numerical stability
            # ll = -0.5 * (innov**2) / self.R_var[i]
            nu = 3.0  # degrees of freedom for Student-t
            scale2 = self.R_var[i]
            ll = -0.5 * (nu + 1.0) * np.log1p((innov**2) / (nu * scale2))
            # ll = -0.5 * np.abs(innov) / self.R_var[i]  # Using absolute for robustness
            ll -= np.max(ll)  # Subtract max for numerical stability
            like = np.exp(ll) + 1e-300  # Add small epsilon to avoid zero
            
            # Update weights (importance sampling)
            self.weights_pf[i] *= like
            self.weights_pf[i] /= np.sum(self.weights_pf[i])  # Normalize

            # ===== RESAMPLE STEP =====
            # Check if resampling is needed based on ESS
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """
        Get current weight estimate (weighted average of particles).
        
        Returns:
        --------
        list of arrays : Weight estimates for each neuron
        """
        return [np.average(self.particles[i], weights=self.weights_pf[i], axis=0) 
                for i in range(self.num_neurons)]
    
    def get_parameters_info(self):
        """
        Get information about Q and R parameters per neuron.
        
        Returns:
        --------
        str : Formatted parameter information
        """
        info = "\nParámetros del Particle Filter:\n"
        info += "=" * 60 + "\n"
        info += f"{'Neurona':<10} {'Q_std':<15} {'R_std':<15} {'R_var':<15}\n"
        info += "-" * 60 + "\n"
        for i in range(self.num_neurons):
            info += f"{i:<10} {self.Q_std[i]:<15.4f} {self.R_std[i]:<15.4f} {self.R_var[i]:<15.6f}\n"
        info += "=" * 60 + "\n"
        info += f"Número de partículas: {self.n_particles}\n"
        info += f"Umbral ESS: {self.ess_threshold:.2f}\n"
        info += "=" * 60 + "\n"
        return info
    
    def get_statistics(self):
        """
        Get current filter statistics.
        
        Returns:
        --------
        dict : Dictionary with ESS and other statistics per neuron
        """
        stats = {
            'ess': [self._ess(self.weights_pf[i]) for i in range(self.num_neurons)],
            'ess_ratio': [self._ess(self.weights_pf[i]) / self.n_particles for i in range(self.num_neurons)],
            'max_weight': [np.max(self.weights_pf[i]) for i in range(self.num_neurons)],
            'min_weight': [np.min(self.weights_pf[i]) for i in range(self.num_neurons)]
        }
        return stats

# ============================================================
# Display available configurations
# ============================================================
print_rhonn_configs()


# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # 🔧 SET RHONN CONFIGURATION HERE
    # ============================================================
    # Choose configuration: 1, 2, 3, 4, or 'custom'
    # - Config 1: Interactions + Quadratic (6 features)
    # - Config 2: Sigmoids + Linear + Bias (7 features)
    # - Config 3: Full Combination (13 features)
    # - Config 4: Compact Interactions + Cubic (4 features)
    # - Config 'custom': Define your own in rhonn_config_custom()
    
    RHONN_CONFIG_ID = 4  # 👈 CHANGE THIS TO SELECT CONFIGURATION
    num_weights_per_neuron = set_rhonn_config(RHONN_CONFIG_ID)
    
    # ============================================================
    
    # --- Reproducibility seed ---
    SEED = np.random.randint(0, 10000)
    np.random.seed(SEED)
    print(f"🎲 Semilla aleatoria (seed): {SEED}")
    print("   (Para reproducibilidad de resultados)\n")
    
    # --- Simulation settings ---
    n_steps = 1000 # More steps for Lorenz to see the attractor
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'cauchy'  # 'laplacian' | 'uniform' | 'gaussian' | 'cauchy'
    process_noise_std = 0.1

    measurement_noise_std = 0.1

    Q_std_per_neuron = [5.0, 6.0, 6.0]  # x, y, z tienen diferentes niveles de ruido de proceso
    R_std_per_neuron = [0.001, 0.001, 0.001]  # x, y, z tienen diferentes niveles de ruido de medición

    # --- True system init ---
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [1.0, 1.0, 1.0] # Initial condition off-center
    
    # --- RHONN config ---
    num_neurons = 3    # x, y, z

    num_particles = 1000

    # --- Initial weights ---
    common_initial_weights = [np.random.uniform(-1.0, 1.0, num_weights_per_neuron) for _ in range(num_neurons)]
    
    print("📊 Pesos iniciales de RHONN:")
    for i, weights in enumerate(common_initial_weights):
        print(f"   Neurona {i} ({['x', 'y', 'z'][i]}): {weights}")
    print()

    # --- Instantiate Trainers ---
    ekf_trainer = EKF_RHONN_Trainer(num_neurons, num_weights_per_neuron, initial_weights=common_initial_weights, eta=1.0, Q_init=1.0e-3, R_init=1.0e-6, P_init=1.0)
    pf_trainer = PF_RHONN_Trainer(num_neurons, num_weights_per_neuron, n_particles=num_particles, initial_weights=common_initial_weights, Q_std=Q_std_per_neuron, R_std=R_std_per_neuron, ess_threshold=0.5*num_particles)

    # Initialize Particle Filter cloud
    for i in range(num_neurons):
        pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles, 1))
        pf_trainer.particles[i] += np.random.randn(pf_trainer.n_particles, num_weights_per_neuron) * 0.01

    # Storage
    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]
    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]

    print("Starting Lorenz Attractor simulation...")
    for k in range(n_steps - 1):
        # 1) Evolve true system
        x_true[k+1] = plant(x_true[k], 0, dt, process_noise_type, process_noise_std) + np.random.laplace(0, measurement_noise_std, size=3)

        # Build series-parallel features from measured state at time k
        chi_k = x_true[k]
        chi_kp1 = x_true[k+1]

        # 2) EKF
        ekf_trainer.update(chi_kp1, chi_k, x_hat_ekf[k])
        # Prediction for next step (Series-Parallel)
        z_ekf = construct_z_vector(chi_k)
        for i in range(3): x_hat_ekf[k+1, i] = np.dot(ekf_trainer.weights[i], z_ekf)

        # 3) PF
        pf_trainer.update(chi_kp1, chi_k, x_hat_pf[k])
        est_w_pf = pf_trainer.get_estimate()
        z_pf = construct_z_vector(chi_k)
        for i in range(3): x_hat_pf[k+1, i] = np.dot(est_w_pf[i], z_pf)

        if k % 200 == 0:
            print(f"Step {k}/{n_steps}")

    print("Simulación Completa.")

    # ============================================================
# 6) Visualización - Formato Tesis
# ============================================================

# Configuración de formato para tesis
thesis_config = {
    'font_family': 'Computer Modern, serif',
    'font_size': 14,
    'title_font_size': 16,
    'legend_font_size': 13,
    'line_width_true': 2.5,
    'line_width_est': 2.0,
    'plot_width': 1000,
    'plot_height': 500,
    'grid_color': 'rgba(200, 200, 200, 0.3)',
    'grid_width': 0.5
}

# Cálculo de MSE por estado
mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_z_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)
mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
mse_z_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)

mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_z_ekf
mse_total_pf = mse_x_pf + mse_y_pf + mse_z_pf

# Reporte MSE
print("\n" + "="*70)
print("🏆 MEJOR FILTRO: ", end="")
mse_dict = {'EKF': mse_total_ekf, 'PF': mse_total_pf}
best_filter = min(mse_dict, key=mse_dict.get)
print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
for other_filter, mse_value in mse_dict.items():
    if other_filter != best_filter:
        print(f"{other_filter} MSE total: {mse_value:.6f}")
print("="*70)

print("\n--- Comparación de Desempeño (MSE) - Atractor de Lorenz ---")
print(f"EKF MSE x:  {mse_x_ekf:.6f}")
print(f"EKF MSE y:  {mse_y_ekf:.6f}")
print(f"EKF MSE z:  {mse_z_ekf:.6f}")
print(f"PF  MSE x:  {mse_x_pf:.6f}")
print(f"PF  MSE y:  {mse_y_pf:.6f}")
print(f"PF  MSE z:  {mse_z_pf:.6f}")

# 5139


# Gráficas individuales por estado
states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'Estado X', 'y_label': 'x'},
    {'idx': 1, 'var': 'y', 'desc': 'Estado Y', 'y_label': 'y'},
    {'idx': 2, 'var': 'z', 'desc': 'Estado Z', 'y_label': 'z'}
]

for state_info in states_info:
    i = state_info['idx']
    
    fig = go.Figure()
    
    # Estado real (línea negra gruesa)
    fig.add_trace(go.Scatter(
        x=t_history, y=x_true[:, i],
        mode='lines',
        name='Estado Real',
        line=dict(color='#000000', width=thesis_config['line_width_true']),
        showlegend=True
    ))
    
    # Estimación EKF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_ekf[:, i],
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
        showlegend=True
    ))
    
    # Estimación PF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_pf[:, i],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
        showlegend=True
    ))
    
    fig.update_layout(
        title={
            'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Atractor de Lorenz',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title=state_info['y_label'],
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.95)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig.show()

# Espacio de fase 3D (Atractor de Lorenz)
fig_phase = go.Figure()

fig_phase.add_trace(go.Scatter3d(
    x=x_true[:, 0], y=x_true[:, 1], z=x_true[:, 2],
    mode='lines',
    name='Atractor Real',
    line=dict(color='#000000', width=4)
))

fig_phase.add_trace(go.Scatter3d(
    x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], z=x_hat_ekf[:, 2],
    mode='lines',
    name='Estimación EKF-RHONN',
    line=dict(color='#1f77b4', width=3)
))

fig_phase.add_trace(go.Scatter3d(
    x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], z=x_hat_pf[:, 2],
    mode='lines',
    name='Estimación PF-RHONN',
    line=dict(color='#d62728', width=3)
))

fig_phase.update_layout(
    title={
        'text': 'Espacio de Fases 3D - Atractor de Lorenz',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            showline=True,
            linewidth=1.5,
            linecolor='black'
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            showline=True,
            linewidth=1.5,
            linecolor='black'
        ),
        zaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            showline=True,
            linewidth=1.5,
            linecolor='black'
        )
    ),
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.95)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    width=thesis_config['plot_width'],
    height=thesis_config['plot_width'],  # Aspecto cuadrado
    margin=dict(l=40, r=40, t=80, b=40)
)

# fig_phase.show()

# Gráfica de barras comparando MSE
fig_mse = go.Figure()

filters = ['EKF-RHONN', 'PF-RHONN']

fig_mse.add_trace(go.Bar(
    name='Estado x',
    x=filters,
    y=[mse_x_ekf, mse_x_pf],
    marker_color='#636EFA',
    text=[f'{mse_x_ekf:.2e}', f'{mse_x_pf:.2e}'],
    textposition='outside'
))

fig_mse.add_trace(go.Bar(
    name='Estado y',
    x=filters,
    y=[mse_y_ekf, mse_y_pf],
    marker_color='#EF553B',
    text=[f'{mse_y_ekf:.2e}', f'{mse_y_pf:.2e}'],
    textposition='outside'
))

fig_mse.add_trace(go.Bar(
    name='Estado z',
    x=filters,
    y=[mse_z_ekf, mse_z_pf],
    marker_color='#00CC96',
    text=[f'{mse_z_ekf:.2e}', f'{mse_z_pf:.2e}'],
    textposition='outside'
))

fig_mse.update_layout(
    title={
        'text': 'Comparación de Error Cuadrático Medio (MSE)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Error Cuadrático Medio (MSE)',
    yaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.95)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    barmode='group',
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_mse.show()





📋 AVAILABLE RHONN CONFIGURATIONS:

Config 1: Interactions + Quadratic
   Features: 6
   Sample z (for x=[1,2,3]): [0.28865141 0.30157037 0.31584803]...

Config 2: Sigmoids + Linear + Bias
   Features: 7
   Sample z (for x=[1,2,3]): [0.52497919 0.549834   0.57444252]...

Config 3: Full Combination
   Features: 13
   Sample z (for x=[1,2,3]): [0.52497919 0.549834   0.57444252]...

Config 4: Compact Interactions + Cubic
   Features: 4
   Sample z (for x=[1,2,3]): [0.28865141 0.30157037 0.31584803]...

Config custom: Custom Configuration
   Features: 6
   Sample z (for x=[1,2,3]): [0.28865141 0.30157037 0.31584803]...


🔧 RHONN Configuration Set:
   ID: 4
   Name: Compact Interactions + Cubic
   Number of features: 4

🎲 Semilla aleatoria (seed): 6382
   (Para reproducibilidad de resultados)

📊 Pesos iniciales de RHONN:
   Neurona 0 (x): [0.92719017 0.86244487 0.90601269 0.31023039]
   Neurona 1 (y): [-0.97110339 -0.52000887 -0.09841281  0.12249984]
   Neurona 2 (z): [-0.69601684 -0.355889